# 03 — Normalización de variables corporales

## Objetivo

Transformar las coordenadas de MediaPipe para que los movimientos sean
comparables entre videos aunque cambien la posición, escala o distancia
de la persona frente a la cámara.

## Estrategia inicial

- Centrar el cuerpo usando el punto medio de las caderas.
- Utilizar la longitud del torso como escala corporal.
- Conservar las marcas de tiempo.
- Mantener la visibilidad de cada landmark.
- No modificar los datos originales de MediaPipe.

## Limitaciones

La normalización inicial trabajará con coordenadas 2D y 3D relativas.
La orientación corporal y la perspectiva de cámara se estudiarán
posteriormente.

In [ ]:
from pathlib import Path
import json
import math

PROJECT_ROOT = Path.cwd().parent
POSE_DATA_DIR = PROJECT_ROOT / "data" / "derived" / "pose"

pose_files = sorted(POSE_DATA_DIR.glob("*_pose.json"))

print(f"Archivos de pose encontrados: {len(pose_files)}")

for pose_file in pose_files:
    print(pose_file.name)

In [ ]:
selected_pose_file = (
    POSE_DATA_DIR
    / "gPO_sBM_c01_d10_mPO0_ch01_pose.json"
)

with selected_pose_file.open("r", encoding="utf-8") as file:
    pose_data = json.load(file)

print(f"Video: {pose_data['source_video']}")
print(f"Frames: {pose_data['frame_count']}")
print(f"FPS: {pose_data['fps']}")
print(f"Landmarks por frame: {pose_data['landmark_count']}")

In [ ]:
LANDMARK_IDS = {
    "left_shoulder": 11,
    "right_shoulder": 12,
    "left_hip": 23,
    "right_hip": 24,
}

def get_landmark(frame, landmark_id):
    for landmark in frame["landmarks"]:
        if landmark["landmark_id"] == landmark_id:
            return landmark

    return None


first_detected_frame = next(
    frame
    for frame in pose_data["frames"]
    if frame["detected"]
)

for name, landmark_id in LANDMARK_IDS.items():
    landmark = get_landmark(first_detected_frame, landmark_id)
    print(name, landmark)